# Day 3-03｜ByteTrack-to-BEV 整合輸出


## 執行階段提醒

- 這一單元把 Day 1 的 Homography、Day 2 的偵測、Day 3 的 tracking 串成一條完整流程。
- 輸出影片左半邊是原始視角 tracking，右半邊是投影到球場平面的 BEV。


## 課程流程

1. 設定影片、detector、court keypoint model 與 BEV 規格。
2. 執行 ByteTrack-to-BEV 影片輸出。
3. 檢查結構化資料欄位：`track_id`、foot point、BEV 座標。
4. 匯出 CSV，讓後續分析 notebook 可以直接接續使用。


In [ ]:
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT)


## Step 1｜設定輸入來源與輸出路徑


In [ ]:
import pandas as pd

from src.video_utils import display_video_in_notebook
from src.yolo_utils import (
    detector_model_path,
    preferred_court_keypoint_model_path,
    reference_videos,
    write_bytetrack_bev_video,
)

videos = reference_videos(COURSE_ROOT)
if len(videos) < 3:
    raise FileNotFoundError("assets/raw/reference_videos/ 至少需要三支參考影片。")

VIDEO_PATH = videos[2]
DETECTOR_PATH = detector_model_path(COURSE_ROOT)
COURT_MODEL_PATH = preferred_court_keypoint_model_path(COURSE_ROOT)
BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"
OUTPUT_PATH = COURSE_ROOT / "assets" / "results" / "d3_03_bytetrack_bev.mp4"
OUT_CSV = COURSE_ROOT / "assets" / "results" / "d3_03_bytetrack_bev.csv"
START_FRAME = 30
MAX_FRAMES = 90

print("video:", VIDEO_PATH)
print("detector:", DETECTOR_PATH)
print("court model:", COURT_MODEL_PATH)
print("bev spec:", BEV_SPEC_PATH)
print("output video:", OUTPUT_PATH)
print("start frame:", START_FRAME)
print("max frames:", MAX_FRAMES)


## Step 2｜執行整合流程

`write_bytetrack_bev_video()` 內部會依序做四件事：

1. 用 detector + ByteTrack 取得每個球員的 `track_id`。
2. 用 court keypoint model 估計當前 frame 的 Homography。
3. 取每個 bbox 的 foot point（底部中心點）並投影到 BEV。
4. 輸出左右對照的預覽影片與結構化 JSON。


In [ ]:
bev_video, records = write_bytetrack_bev_video(
    video_path=VIDEO_PATH,
    detector_path=DETECTOR_PATH,
    court_model_path=COURT_MODEL_PATH,
    bev_spec_path=BEV_SPEC_PATH,
    output_path=OUTPUT_PATH,
    max_frames=MAX_FRAMES,
    detector_conf=0.25,
    keypoint_conf=0.15,
    anchor_confidence=0.25,
    imgsz=960,
    start_frame=START_FRAME,
)

print("BEV video:", bev_video)
print("BEV json:", bev_video.with_suffix(".json"))
print("rows:", len(records))
display_video_in_notebook(bev_video, loop=True)


## Step 3｜檢查結構化輸出欄位


In [ ]:
track_columns = [
    "frame",
    "track_id",
    "class_name",
    "confidence",
    "bbox_xyxy",
    "foot_x",
    "foot_y",
    "bev_x",
    "bev_y",
    "keypoint_count",
]
tracks = pd.DataFrame(records, columns=track_columns)
tracks.head(10)


## Step 4｜匯出 CSV 並做簡單摘要

這份 CSV 之後可以直接拿去做：

- 單一球員路徑視覺化
- 球員速度估計
- 攻守站位分析
- 與 shooting / event pipeline 串接


In [ ]:
tracks.to_csv(OUT_CSV, index=False)

if tracks.empty:
    summary = pd.DataFrame(columns=["track_id", "frames_seen", "first_frame", "last_frame", "mean_bev_x", "mean_bev_y"])
else:
    summary = (
        tracks.groupby("track_id", dropna=False)
        .agg(
            frames_seen=("frame", "count"),
            first_frame=("frame", "min"),
            last_frame=("frame", "max"),
            mean_bev_x=("bev_x", "mean"),
            mean_bev_y=("bev_y", "mean"),
        )
        .sort_values(["frames_seen", "first_frame"], ascending=[False, True])
        .reset_index()
    )

print("BEV csv:", OUT_CSV)
summary.head(15)


Day 1 到 Day 3 的主流程至此完成：Roboflow 標註格式、YOLO detection、court keypoint Homography、ByteTrack 與 BEV 路徑輸出。輸出流程會在當前 frame keypoint 不足時，優先沿用上一個穩定的 Homography，避免投影影片閃爍太嚴重。


## 本單元產出檔案

- `assets/results/d3_03_bytetrack_bev.mp4`
- `assets/results/d3_03_bytetrack_bev.json`
- `assets/results/d3_03_bytetrack_bev.csv`
